# V14 - Dados e integracoes do projeto

**Objetivo (read-only):** correlacionar o catalogo DMS (data models, views) com as integracoes (Transformations) existentes, sem alterar nada.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v14',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Catalogo DMS

In [ ]:
import pandas as pd

models = client.data_modeling.data_models.list(limit=100)
views = client.data_modeling.views.list(limit=100)
print('data models:', len(models), '| views:', len(views))
models.to_pandas().head()

## 2. Integracoes (Transformations)

In [ ]:
transformations = client.transformations.list(limit=100)
print('transformations:', len(transformations))
tr_df = transformations.to_pandas()
tr_df.head()

## 3. Correlacao por space
Quantas views por space versus quantas transformations existem, lado a lado.

In [ ]:
import matplotlib.pyplot as plt

views_df = views.to_pandas()
if 'space' in views_df.columns and len(views_df):
    by_space = views_df['space'].value_counts().head(8)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.barh(by_space.index.astype(str)[::-1], by_space.values[::-1])
    ax.set_title('Views por space (top 8)')
    ax.set_xlabel('quantidade de views')
    plt.tight_layout()
    plt.show()
else:
    print('Sem views para correlacionar neste ambiente.')

catalog = {
    'data_models': len(models),
    'views': len(views),
    'transformations': len(transformations),
}
print('resumo do projeto:', catalog)

## Mini-exercicio
- Liste os `external_id` das Transformations cujo destino e DMS (`instances`/`nodes`/`edges`).
- Cruze (manualmente) um data model com a Transformation que o popula, se houver.